# 1 · Guainía: BIOMASS L1c tomographic stack → netCDF

**Pipeline:** 0 · Data access → **1 · Stack** → 2 · Tomography → 3 · Forest height → 4 · Polarimetric analysis

This notebook turns the seven BIOMASS Level-1c stack products in `stack_download/` into **one analysis-ready netCDF file**. Notebooks 2–4 read only that file and `Guainia_outputs/run_manifest.json`.

| | |
|---|---|
| Mission phase | Tomographic (TOM), major cycle 3 |
| Swath / track / frame | S1 / T029 / F152, descending |
| Acquisitions | 7 repeat cycles (C01–C07), 2026-04-24 to 2026-05-12, 3-day revisit |
| Location | Guainía, Colombia (≈ 2.85–4.14° N, 69.5–70.35° W) |
| Product | `BIO_S1_STA__1S`: quad-pol single-look complex (β⁰), coregistered onto one primary |

**Steps**
1. list the products and identify the coregistration primary;
2. check one product: files, annotation pixel grid, LUT layers and their axes;
3. write all acquisitions and LUT layers to one netCDF on the common SLC grid;
4. verify the stack: layout, no-data, LUT layers;
5. check the **baseline requirement** for tomography (vertical resolution, height of ambiguity);
6. quick-looks;
7. write the run manifest.

## 1. Configuration
`CROP = None` writes the full frame (20 765 azimuth × 1 315 range samples, ≈ 6 GB of SLC plus compressed LUT layers, a few minutes). A crop is given in product pixel indices, e.g. `{"azimuth": (8000, 12000), "range": (0, 1315)}`.

In [ ]:
import pathlib, sys
SITE_DIR = pathlib.Path.cwd()                       # the Guainia/ folder (notebooks run from here)
if not (SITE_DIR / "stack_download").is_dir():
    raise FileNotFoundError("open the notebook from the Guainia/ folder (stack_download/ not found)")
ROOT = next(p for p in [SITE_DIR, *SITE_DIR.parents] if (p / "biomass_tomo").is_dir())
sys.path.insert(0, str(ROOT))                       # the biomass_tomo package

import re, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from biomass_tomo import l1c
from biomass_tomo import stack as st

SITE = "guainia"
STACK_DIR = SITE_DIR / "stack_download"
POLARISATIONS = ("HH", "HV", "VH", "VV")
CROP = None                      # None = full frame; or {"azimuth": (a0, a1), "range": (r0, r1)}
MIN_ACQUISITIONS = 3             # baseline requirement: minimum number of acquisitions
Z_RANGE = (-30, 80)              # height interval to image [m]; must fit inside the height of ambiguity
OVERWRITE = False                # rebuild the netCDF even if it exists

# optional reference rasters (any CRS) for notebooks 3-4; none exist for Guainía yet
REFERENCE_CHM = None             # e.g. ("chm.tif", "band name or index")
REFERENCE_AGBD = None            # e.g. ("agbd.tif", 1)

OUT_DIR = SITE_DIR / "Guainia_outputs"
OUT_DIR.mkdir(exist_ok=True)

## 2. Products
Each product folder holds
- `measurement/`: `*_i_abs.tiff` and `*_i_phase.tiff` (4 bands, HH HV VH VV), so $s = |s|\,e^{j\varphi}$;
- `annotation_coregistered/`: the annotation XML and the **LUT file** `*_lut.nc`, both on the primary's grid;
- `annotation_primary/`: the acquisition's own annotation before coregistration;
- `preview/`: quick-look PNG/KML.

The coregistration primary is the acquisition with zero normal baseline. Its LUT has no `/interferometry` group, because its coherence with itself is trivially 1.

In [ ]:
product_dirs = l1c.find_product_dirs(STACK_DIR)
rows = []
for p in product_dirs:
    ann = l1c.product_file(p, "annotation", "*_annot.xml")
    get = lambda tag: l1c.annotation_value(ann, (tag,))
    rows.append({"cycle": int(re.search(r"_C(\d\d)_", p.name).group(1)),
                 "start": pd.to_datetime(re.search(r"_1S_(\d{8}T\d{6})_", p.name).group(1)),
                 "primary": l1c.is_primary(p),
                 "normal_baseline_m": float(get("normalBaseline")),
                 "baseline_order": int(get("baselineOrderingIndex")),
                 "coregistration": get("coregistrationMethod"),
                 "product": p.name})
products = pd.DataFrame(rows)
PRIMARY = product_dirs[int(products.index[products.primary][0])]
STACK_KEY = re.search(r"BIO_(S\d)_.*_(M\d\d)_C\d\d_(T\d{3})_(F\d{3})_(\d\d)_", PRIMARY.name).groups()
STACK_KEY = "_".join(STACK_KEY[:4]) + f"_v{STACK_KEY[4]}"
print("stack:", STACK_KEY, "| primary:", PRIMARY.name)
products

## 3. Product check
`inspect_product` lists the files and reads the pixel grid from the annotation (`firstSampleSlantRangeTime`, `firstLineAzimuthTime`, `rangeTimeInterval`, `azimuthTimeInterval`, pixel spacings). It also lists every variable in the LUT file.

All LUT layers are on a coarse grid (1731 × 658) whose axes are **relative azimuth time** (s since the first line) and **two-way slant-range time**. `biomass_tomo.l1c` interpolates them bilinearly onto the SLC grid. The layers written to the stack:

| stack variable | LUT path | per acquisition | used for |
|---|---|---|---|
| `dem_phases` | `/coregistration/flatteningPhaseScreen` | ✓ | removes the DEM (topographic) phase |
| `calibration_phases` | `/skpPhaseCalibration/skpCalibrationPhaseScreen` | ✓ | SKP ground-phase calibration: ground → $z=0$ |
| `insar_wavenumbers` | `/coregistration/waveNumbers` | ✓ | vertical wavenumber $k_z$: tomographic steering |
| `calibration_quality`, `coregistration_quality` | `…Quality` | ✓ | quality flags |
| `lut_coherence`, `lut_interferogram` | `/interferometry/…` | ✓ | L1c coherence with the primary (1 / 0 for the primary) |
| `noise_HH` … `noise_VV` | `/denoising/denoising{pol}` | ✓ | thermal noise power (β⁰), not removed in L1c |
| `incidence_angle`, `range_slope`, `azimuth_slope` | `/geometry/…` | | ground-range windows, terrain |
| `dem`, `latitude`, `longitude` | `/geometry/…` | | geolocation |
| `sigma_nought_factor`, `gamma_nought_factor` | `/radiometry/…` | | β⁰ → σ⁰, γ⁰ (power factors) |

In [ ]:
l1c.inspect_product(PRIMARY)

### Frame footprint
The frame outline from the LUT geolocation, with the processor's Pauli quick-look of the primary.

In [ ]:
import folium
az, rg, lat_lut, *_ = l1c.lut_axes(l1c.product_file(PRIMARY, "annotation", "*_lut.nc"), "/geometry/latitude")
_, _, lon_lut, *_ = l1c.lut_axes(l1c.product_file(PRIMARY, "annotation", "*_lut.nc"), "/geometry/longitude")
edge = lambda a: np.r_[a[0, :], a[:, -1], a[-1, ::-1], a[::-1, 0]]
outline = list(zip(edge(lat_lut), edge(lon_lut)))
FRAME_BBOX = (float(np.nanmin(lon_lut)), float(np.nanmin(lat_lut)), float(np.nanmax(lon_lut)), float(np.nanmax(lat_lut)))
print("frame bbox (W, S, E, N):", tuple(round(v, 4) for v in FRAME_BBOX))

m = folium.Map(location=[np.nanmean(lat_lut), np.nanmean(lon_lut)], zoom_start=8)
folium.Polygon(outline, color="red", weight=2, fill=False, tooltip=STACK_KEY).add_to(m)
folium.Marker([lat_lut[0, 0], lon_lut[0, 0]], tooltip="first line, near range").add_to(m)
m

In [ ]:
ql = sorted((PRIMARY / "preview").glob("*_l1c_ql_pauli.png"))
if ql:
    img = plt.imread(ql[0])
    fig, ax = plt.subplots(figsize=(4, 9), constrained_layout=True)
    ax.imshow(img, aspect="auto"); ax.set_title("Pauli quick-look (processor)", fontsize=9); ax.axis("off")
    plt.show()

## 4. Build the stack netCDF
Every acquisition is read on the same pixel grid, because the products are already coregistered. Layout (`biomass_tomo.l1c.build_stack`, identical to the course stack):

| variable | dimensions | content |
|---|---|---|
| `slc_real`, `slc_imag` | (polarisation, acquisition, azimuth, range) | β⁰ SLC, $s = $ `slc_real + 1j*slc_imag`, no-data → 0 |
| per-acquisition LUT layers | (acquisition, azimuth, range) | see the table above |
| single LUT layers | (azimuth, range) | geometry, radiometry |
| `azimuth`, `range` | | along-track position, slant range [m] |
| `azimuth_line`, `range_sample` | | pixel indices in the original frame |
| `temporal_baseline`, `acquisition_time`, `product_id` | (acquisition) | |

LUT values are stored **as delivered**. Their sign convention is determined from the data in notebook 2.

In [ ]:
STACK_PATH = OUT_DIR / f"{SITE}_{STACK_KEY}_stack.nc"
if STACK_PATH.exists() and not OVERWRITE:
    print("exists, not rebuilt:", STACK_PATH)
else:
    t0 = time.time()
    last = [0.0]
    def progress(msg):
        if msg.startswith("  SLC") and time.time() - last[0] < 20:
            return
        last[0] = time.time()
        print(f"[{time.time() - t0:6.0f} s] {msg}", flush=True)
    l1c.build_stack(STACK_DIR, STACK_PATH, crop=CROP, polarisations=POLARISATIONS, site=SITE, progress=progress)
    print(f"done in {time.time() - t0:.0f} s")
print(f"{STACK_PATH.name}: {STACK_PATH.stat().st_size / 1e9:.2f} GB")

## 5. Verification
Dimensions, global attributes, every variable with its source LUT and units, and the no-data count per channel.

In [ ]:
ds = st.open_stack(STACK_PATH)
print(dict(ds.sizes))
for k in ("title", "processor", "swath", "polarisation_sequence", "crop_azimuth", "crop_range", "history"):
    print(f"  {k:22s} {ds.attrs.get(k)}")
pd.DataFrame([{"variable": n, "dims": ", ".join(v.dims), "dtype": str(v.dtype), "units": v.attrs.get("units", ""),
               "source": v.attrs.get("source", "")} for n, v in ds.data_vars.items()]).set_index("variable")

In [ ]:
nodata = pd.DataFrame(ds.nodata_pixels.values / (ds.sizes["azimuth"] * ds.sizes["range"]) * 100,
                      index=ds.polarisation.values, columns=[f"C{c:02d}" for c in ds.acquisition.values])
print("no-data pixels [% of frame] (unfocused samples at the frame edges, see the black border in the quick-looks)")
nodata.round(2)

### LUT layers on the SLC grid
The phase layers are shown for the primary and for the acquisition with the largest $|k_z|$. For the primary, the flattening phase and $k_z$ must be zero. The flattening phase grows with baseline and follows topography. The SKP screen is the ground phase estimated per pixel from the Sum-of-Kronecker-Products decomposition. It combines the slowly varying residual errors (orbit, ionosphere, troposphere) with fine structure that follows land cover and DEM errors, which is why it is not smooth.

In [ ]:
REF = st.reference_index(ds)
kz_scene = ds.insar_wavenumbers.isel(azimuth=slice(None, None, 200), range=slice(None, None, 50)).mean(("azimuth", "range")).values
FAR = int(np.argmax(np.abs(kz_scene)))
dec = dict(azimuth=slice(None, None, 20), range=slice(None, None, 2))
ext = [0, ds.sizes["azimuth"], 0, ds.sizes["range"]]
layers = [("dem_phases", REF, "twilight"), ("dem_phases", FAR, "twilight"), ("calibration_phases", FAR, "twilight"),
          ("insar_wavenumbers", FAR, "viridis"), ("incidence_angle", None, "viridis"), ("dem", None, "terrain"),
          ("sigma_nought_factor", None, "viridis"), ("noise_HV", REF, "magma")]
fig, axes = plt.subplots(len(layers), 1, figsize=(11, 1.55 * len(layers)), sharex=True, constrained_layout=True)
for ax, (name, ia, cmap) in zip(axes, layers):
    da = ds[name].isel(**dec) if ia is None else ds[name].isel(acquisition=ia, **dec)
    v = da.values.T
    if name in ("dem_phases", "calibration_phases"):
        v = np.angle(np.exp(1j * v)); kw = dict(vmin=-np.pi, vmax=np.pi)
    elif name == "noise_HV":
        v = 10 * np.log10(v); kw = {}
    else:
        kw = dict(vmin=np.nanpercentile(v, 1), vmax=np.nanpercentile(v, 99))
    im = ax.imshow(v, aspect="auto", origin="lower", cmap=cmap, extent=ext, interpolation="none", **kw)
    lab = name + ("" if ia is None else f"  C{int(ds.acquisition[ia]):02d}") + (" (wrapped)" if "phases" in name else "")
    ax.set_title(lab + (" [dB]" if name == "noise_HV" else f" [{ds[name].attrs.get('units', '')}]"), fontsize=8, loc="left")
    ax.set_ylabel("range", fontsize=8); fig.colorbar(im, ax=ax, pad=0.01, aspect=8)
axes[-1].set_xlabel("azimuth line")
plt.show()

## 6. Baseline requirement
Tomography needs several acquisitions with different vertical wavenumbers $k_z^i$, where $k_z^i$ converts height into interferometric phase with the primary ($\phi = k_z z$).
- Rayleigh **vertical resolution**: $\delta z = 2\pi/\Delta k_z$, with $\Delta k_z$ the total span of the $k_z$ values.
- **Height of ambiguity**: $z_{amb} = 2\pi/\min_i |k_z^{i+1}-k_z^i|$. Heights further apart than $z_{amb}$ fold onto each other, so `Z_RANGE` must fit inside it.

$k_z$ varies with incidence angle, so both are also shown across range.

In [ ]:
report = st.baseline_report(ds, min_acquisitions=MIN_ACQUISITIONS, z_extent=Z_RANGE)
for k in ("n_acquisitions", "cycles", "kz_mean_rad_per_m", "kz_span", "vertical_resolution_m",
          "vertical_resolution_m_p5_p95", "height_of_ambiguity_m", "reference_acquisition_index", "tomography_ok", "problems"):
    v = report[k]
    print(f"{k:30s} {np.round(v, 4).tolist() if isinstance(v, list) and v and isinstance(v[0], float) else v}")

kz_rg = ds.insar_wavenumbers.isel(azimuth=slice(None, None, 100)).mean("azimuth").values      # (acq, range)
inc_rg = ds.incidence_angle.isel(azimuth=slice(None, None, 100)).mean("azimuth").values
kz_sorted = np.sort(kz_rg, axis=0)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), constrained_layout=True)
axes[0].plot(report["temporal_baseline_days"], report["kz_mean_rad_per_m"], "o-")
axes[0].plot(report["temporal_baseline_days"][REF], report["kz_mean_rad_per_m"][REF], "ro", mfc="none", ms=12, label="primary")
axes[0].set_xlabel("temporal baseline [days]"); axes[0].set_ylabel("mean $k_z$ [rad/m]"); axes[0].legend()
for i in range(kz_rg.shape[0]):
    axes[1].plot(inc_rg, kz_rg[i], label=f"C{int(ds.acquisition[i]):02d}")
axes[1].set_xlabel("incidence angle [deg]"); axes[1].set_ylabel("$k_z$ [rad/m]"); axes[1].legend(fontsize=7, ncol=2)
axes[2].plot(inc_rg, 2 * np.pi / (kz_sorted[-1] - kz_sorted[0]), label=r"resolution $\delta z$")
axes[2].plot(inc_rg, 2 * np.pi / np.min(np.diff(kz_sorted, axis=0), axis=0), label=r"ambiguity $z_{amb}$")
axes[2].axhspan(0, Z_RANGE[1] - Z_RANGE[0], color="0.9", label="Z_RANGE extent")
axes[2].set_xlabel("incidence angle [deg]"); axes[2].set_ylabel("[m]"); axes[2].legend(fontsize=8)
for ax in axes: ax.grid(alpha=0.3)
plt.show()
if not report["tomography_ok"]:
    raise RuntimeError("Stack fails the baseline requirement: " + "; ".join(report["problems"]))

## 7. Quick-looks
β⁰ intensity of the primary in HH, HV and VV, and a Pauli RGB (R $|HH-VV|^2$, G $|HV+VH|^2$, B $|HH+VV|^2$). All are averaged over 20 × 2 samples (≈ 130 m × 95 m on the ground) to suppress speckle.

In [ ]:
def block_mean(a, f=(2, 20)):                     # a: (range, azimuth)
    nr, nx = (a.shape[0] // f[0]) * f[0], (a.shape[1] // f[1]) * f[1]
    return a[:nr, :nx].reshape(nr // f[0], f[0], nx // f[1], f[1]).mean(axis=(1, 3))

load = lambda pol: st.course_layout(ds.slc_real.sel(polarisation=pol).isel(acquisition=REF)) + \
                   1j * st.course_layout(ds.slc_imag.sel(polarisation=pol).isel(acquisition=REF))
S = {p: load(p) for p in POLARISATIONS}
I = {"HH": block_mean(np.abs(S["HH"]) ** 2), "HV": block_mean(np.abs(S["HV"]) ** 2), "VV": block_mean(np.abs(S["VV"]) ** 2)}
pauli = np.stack([block_mean(np.abs(S["HH"] - S["VV"]) ** 2 / 2), block_mean(np.abs(S["HV"] + S["VH"]) ** 2 / 2),
                  block_mean(np.abs(S["HH"] + S["VV"]) ** 2 / 2)], axis=-1)
del S
ext = [0, ds.sizes["azimuth"], 0, ds.sizes["range"]]
fig, axes = plt.subplots(4, 1, figsize=(11, 8.5), sharex=True, constrained_layout=True)
for ax, (p, v) in zip(axes, I.items()):
    with np.errstate(divide="ignore"):
        db = 10 * np.log10(v)
    im = ax.imshow(db, aspect="auto", origin="lower", cmap="gray", extent=ext,
                   vmin=np.nanpercentile(db[np.isfinite(db)], 2), vmax=np.nanpercentile(db[np.isfinite(db)], 98))
    ax.set_title(f"β⁰ {p} [dB], C{int(ds.acquisition[REF]):02d}", fontsize=8, loc="left"); ax.set_ylabel("range")
    fig.colorbar(im, ax=ax, pad=0.01, aspect=8)
a = np.sqrt(pauli); a = np.clip(a / np.nanpercentile(a, 98, axis=(0, 1)), 0, 1)
axes[3].imshow(a, aspect="auto", origin="lower", extent=ext)
axes[3].set_title("Pauli RGB (R: HH−VV, G: HV+VH, B: HH+VV)", fontsize=8, loc="left"); axes[3].set_ylabel("range")
axes[3].set_xlabel("azimuth line")
plt.show()

## 8. Run manifest
`Guainia_outputs/run_manifest.json` holds everything notebooks 2–4 need: the stack path, the analysis crop (default: the whole stack, in stack pixel indices `[az0, az1, rg0, rg1]`), the height range, the baseline report and the optional reference rasters. Each later notebook adds its own results to it.

In [ ]:
MANIFEST = OUT_DIR / "run_manifest.json"
st.write_manifest(
    MANIFEST, site=SITE, stack_key=STACK_KEY, stack_path=str(STACK_PATH), stack_dir=str(STACK_DIR),
    product_ids=[str(p) for p in ds.product_id.values], primary=PRIMARY.name, frame_bbox=FRAME_BBOX,
    crop=[0, int(ds.sizes["azimuth"]), 0, int(ds.sizes["range"])], z_range=list(Z_RANGE), baseline=report,
    out_dir=str(OUT_DIR), reference={"chm": REFERENCE_CHM, "agbd": REFERENCE_AGBD})
ds.close()
print("written", MANIFEST)